# Gap Junction Coupling Study: Analysis Notebook
## CNS F25 Project 2 - Rod Network Dynamic Range

This notebook analyzes the gap junction coupling sweep experiment, computing:
- **EC50**: half-maximal response intensity
- **Slope**: Hill coefficient / dynamic range compression
- **Saturation**: plateau response level
- **SNR**: signal-to-noise ratio across intensities
- **Statistical robustness**: bootstrap CIs and permutation tests

Based on Publio et al. (2009) rod network model with LN bipolar readout.

## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit
from scipy import stats
import glob
import os

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

print("Libraries imported successfully")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 2. Load Simulation Results

In [ ]:
def load_coupling_data():
    """Load all coupling sweep data files"""
    
    data_files = sorted(glob.glob('results_coupling_sweep_g*.dat'))
    print(f"Found {len(data_files)} data files")
    
    all_data = []
    
    for fname in data_files:
        # Extract gap conductance from filename
        g_gap = float(fname.split('_g')[1].split('_')[0])
        
        # Load data
        df = pd.read_csv(fname, sep='\t', comment='#', 
                        names=['Intensity', 'Trial', 'SpikeCount', 'FiringRate', 'LN_Response'])
        df['g_gap'] = g_gap
        
        all_data.append(df)
    
    # Combine all data
    full_df = pd.concat(all_data, ignore_index=True)
    
    print(f"Loaded {len(full_df)} trials")
    print(f"Gap conductance range: {full_df['g_gap'].min():.2f} - {full_df['g_gap'].max():.2f} nS")
    print(f"Intensity range: {full_df['Intensity'].min():.1f} - {full_df['Intensity'].max():.1f} pA")
    
    return full_df

# Load data
df = load_coupling_data()
df.head()

## 3. Dose-Response Curve Fitting (EC50 and Slope)

In [ ]:
def hill_function(x, R_max, EC50, n, baseline):
    """Hill equation for dose-response curves
    
    Parameters:
    - R_max: maximum response
    - EC50: half-maximal stimulus intensity
    - n: Hill coefficient (slope)
    - baseline: minimum response
    """
    return baseline + (R_max - baseline) * x**n / (EC50**n + x**n)

def fit_dose_response(intensities, responses):
    """Fit Hill equation to dose-response data
    
    Returns: (R_max, EC50, n, baseline), pcov
    """
    # Initial guess
    R_max_init = np.max(responses)
    baseline_init = np.min(responses)
    EC50_init = intensities[np.argmin(np.abs(responses - R_max_init/2))]
    n_init = 1.0
    
    p0 = [R_max_init, EC50_init, n_init, baseline_init]
    
    # Bounds to keep parameters realistic
    bounds = ([0, 0, 0.1, 0], 
              [np.inf, 200, 10, R_max_init])
    
    try:
        popt, pcov = curve_fit(hill_function, intensities, responses, 
                               p0=p0, bounds=bounds, maxfev=10000)
        return popt, pcov
    except:
        return None, None

# Compute mean responses per (g_gap, intensity)
df_mean = df.groupby(['g_gap', 'Intensity'])['FiringRate'].agg(['mean', 'std', 'count']).reset_index()
df_mean['sem'] = df_mean['std'] / np.sqrt(df_mean['count'])

print("Mean responses computed")
print(f"Conditions: {len(df_mean)} (g_gap × intensity combinations)")
df_mean.head(10)

In [ ]:
# Fit dose-response curves for each coupling value
coupling_metrics = []

for g_gap in sorted(df_mean['g_gap'].unique()):
    subset = df_mean[df_mean['g_gap'] == g_gap].sort_values('Intensity')
    
    intensities = subset['Intensity'].values
    responses = subset['mean'].values
    
    # Remove zero intensity (causes issues with Hill fit)
    mask = intensities > 0
    intensities = intensities[mask]
    responses = responses[mask]
    
    # Fit Hill curve
    popt, pcov = fit_dose_response(intensities, responses)
    
    if popt is not None:
        R_max, EC50, n, baseline = popt
        
        # Estimate saturation point (95% of max)
        saturation_intensity = EC50 * ((0.95 * (R_max - baseline) / (0.05 * (R_max - baseline)))**(1/n))
        
        coupling_metrics.append({
            'g_gap': g_gap,
            'R_max': R_max,
            'EC50': EC50,
            'Hill_n': n,
            'baseline': baseline,
            'saturation_intensity': saturation_intensity,
            'dynamic_range': R_max - baseline
        })

metrics_df = pd.DataFrame(coupling_metrics)
print(f"Fitted {len(metrics_df)} dose-response curves")
print("\nSummary of EC50 and Hill coefficient:")
print(metrics_df[['g_gap', 'EC50', 'Hill_n', 'R_max']].describe())
metrics_df.head()

## 4. Signal-to-Noise Ratio (SNR) Analysis

In [ ]:
# Compute SNR for each condition
df_mean['SNR'] = df_mean['mean'] / df_mean['std']
df_mean['SNR'] = df_mean['SNR'].replace([np.inf, -np.inf], np.nan)

# Add SNR at EC50 to metrics
for idx, row in metrics_df.iterrows():
    g_gap = row['g_gap']
    EC50 = row['EC50']
    
    # Find intensity closest to EC50
    subset = df_mean[df_mean['g_gap'] == g_gap]
    closest_idx = (subset['Intensity'] - EC50).abs().idxmin()
    snr_at_ec50 = subset.loc[closest_idx, 'SNR']
    
    metrics_df.at[idx, 'SNR_at_EC50'] = snr_at_ec50

# Compute area under SNR curve (AUC-SNR) as overall metric
snr_auc = []
for g_gap in sorted(df_mean['g_gap'].unique()):
    subset = df_mean[(df_mean['g_gap'] == g_gap) & (df_mean['Intensity'] > 0)]
    subset = subset.sort_values('Intensity')
    
    # Integrate using trapezoidal rule
    auc = np.trapz(subset['SNR'].fillna(0), subset['Intensity'])
    snr_auc.append({'g_gap': g_gap, 'SNR_AUC': auc})

snr_auc_df = pd.DataFrame(snr_auc)
metrics_df = metrics_df.merge(snr_auc_df, on='g_gap')

print("SNR metrics computed")
print("\nSNR at EC50 and total AUC:")
print(metrics_df[['g_gap', 'SNR_at_EC50', 'SNR_AUC']].describe())
metrics_df.head()

## 5. Bootstrap Confidence Intervals

In [ ]:
def bootstrap_ec50(df, g_gap, n_bootstrap=1000, random_state=42):
    """Bootstrap EC50 estimate with confidence intervals"""
    
    np.random.seed(random_state)
    
    subset = df[(df['g_gap'] == g_gap) & (df['Intensity'] > 0)]
    intensities = subset['Intensity'].unique()
    
    ec50_samples = []
    
    for _ in range(n_bootstrap):
        # Resample trials within each intensity
        boot_responses = []
        for intensity in intensities:
            trials = subset[subset['Intensity'] == intensity]['FiringRate'].values
            boot_sample = np.random.choice(trials, size=len(trials), replace=True)
            boot_responses.append(boot_sample.mean())
        
        boot_responses = np.array(boot_responses)
        
        # Fit Hill curve
        popt, _ = fit_dose_response(intensities, boot_responses)
        if popt is not None:
            ec50_samples.append(popt[1])  # EC50 is second parameter
    
    ec50_samples = np.array(ec50_samples)
    ec50_samples = ec50_samples[np.isfinite(ec50_samples)]
    
    if len(ec50_samples) > 0:
        ci_lower = np.percentile(ec50_samples, 2.5)
        ci_upper = np.percentile(ec50_samples, 97.5)
        return ci_lower, ci_upper
    else:
        return np.nan, np.nan

# Compute bootstrap CIs for select coupling values
test_coupling_values = [0.0, 1.0, 2.5, 5.0]

print("Computing bootstrap confidence intervals (this may take a minute)...")
for g_gap in test_coupling_values:
    if g_gap in metrics_df['g_gap'].values:
        ci_lower, ci_upper = bootstrap_ec50(df, g_gap, n_bootstrap=500)
        
        # Add to metrics
        idx = metrics_df[metrics_df['g_gap'] == g_gap].index[0]
        metrics_df.at[idx, 'EC50_CI_lower'] = ci_lower
        metrics_df.at[idx, 'EC50_CI_upper'] = ci_upper
        
        print(f"g={g_gap:.1f} nS: EC50 = {metrics_df.at[idx, 'EC50']:.1f} pA, "
              f"95% CI = [{ci_lower:.1f}, {ci_upper:.1f}]")

print("\nBootstrap analysis complete")

## 6. Permutation Tests (Statistical Significance)

In [ ]:
def permutation_test_ec50(df, g_gap1, g_gap2, n_permutations=1000, random_state=42):
    """Test if EC50 differs significantly between two coupling values"""
    
    np.random.seed(random_state)
    
    # Get data for both conditions
    data1 = df[(df['g_gap'] == g_gap1) & (df['Intensity'] > 0)]
    data2 = df[(df['g_gap'] == g_gap2) & (df['Intensity'] > 0)]
    
    # Observed difference
    ec50_1 = metrics_df[metrics_df['g_gap'] == g_gap1]['EC50'].values[0]
    ec50_2 = metrics_df[metrics_df['g_gap'] == g_gap2]['EC50'].values[0]
    observed_diff = abs(ec50_1 - ec50_2)
    
    # Permutation test
    null_diffs = []
    combined_data = pd.concat([data1, data2])
    
    for _ in range(n_permutations):
        # Shuffle condition labels
        shuffled = combined_data.copy()
        shuffled['g_gap'] = np.random.permutation(shuffled['g_gap'].values)
        
        # Recompute EC50 for each group
        for g_gap in [g_gap1, g_gap2]:
            subset = shuffled[shuffled['g_gap'] == g_gap]
            intensities = subset.groupby('Intensity')['FiringRate'].mean()
            
            if len(intensities) > 3:
                popt, _ = fit_dose_response(intensities.index.values, intensities.values)
                if popt is not None:
                    if g_gap == g_gap1:
                        perm_ec50_1 = popt[1]
                    else:
                        perm_ec50_2 = popt[1]
        
        try:
            null_diffs.append(abs(perm_ec50_1 - perm_ec50_2))
        except:
            pass
    
    null_diffs = np.array(null_diffs)
    p_value = np.sum(null_diffs >= observed_diff) / len(null_diffs)
    
    return observed_diff, p_value

# Test differences between low, medium, and high coupling
print("Permutation tests for EC50 differences:\n")

comparisons = [
    (0.0, 1.0, "No coupling vs. Low coupling"),
    (1.0, 2.5, "Low coupling vs. Medium coupling"),
    (2.5, 5.0, "Medium coupling vs. High coupling"),
    (0.0, 5.0, "No coupling vs. High coupling")
]

for g1, g2, label in comparisons:
    if g1 in metrics_df['g_gap'].values and g2 in metrics_df['g_gap'].values:
        diff, p_val = permutation_test_ec50(df, g1, g2, n_permutations=500)
        print(f"{label}:")
        print(f"  EC50 difference: {diff:.1f} pA, p = {p_val:.3f}")
        print()

## 7. Visualization: Dose-Response Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Select representative coupling values to plot
plot_coupling = [0.0, 1.0, 2.5, 5.0]
colors = plt.cm.viridis(np.linspace(0, 1, len(plot_coupling)))

# Panel A: Dose-response curves
ax = axes[0, 0]
for idx, g_gap in enumerate(plot_coupling):
    subset = df_mean[df_mean['g_gap'] == g_gap]
    subset = subset[subset['Intensity'] > 0].sort_values('Intensity')
    
    ax.errorbar(subset['Intensity'], subset['mean'], yerr=subset['sem'],
                marker='o', label=f'g={g_gap:.1f} nS', color=colors[idx], alpha=0.7)
    
    # Plot fitted Hill curve
    if g_gap in metrics_df['g_gap'].values:
        params = metrics_df[metrics_df['g_gap'] == g_gap].iloc[0]
        x_fit = np.linspace(1, 100, 200)
        y_fit = hill_function(x_fit, params['R_max'], params['EC50'], 
                             params['Hill_n'], params['baseline'])
        ax.plot(x_fit, y_fit, '--', color=colors[idx], alpha=0.5)

ax.set_xlabel('Flash Intensity (pA)')
ax.set_ylabel('Firing Rate (Hz)')
ax.set_title('A. Dose-Response Curves')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel B: Log-log dose-response
ax = axes[0, 1]
for idx, g_gap in enumerate(plot_coupling):
    subset = df_mean[df_mean['g_gap'] == g_gap]
    subset = subset[subset['Intensity'] > 0].sort_values('Intensity')
    
    ax.loglog(subset['Intensity'], subset['mean'], 
              marker='o', label=f'g={g_gap:.1f} nS', color=colors[idx], alpha=0.7)

ax.set_xlabel('Flash Intensity (pA)')
ax.set_ylabel('Firing Rate (Hz)')
ax.set_title('B. Log-Log Dose-Response')
ax.legend()
ax.grid(True, alpha=0.3, which='both')

# Panel C: EC50 vs. coupling
ax = axes[1, 0]
ax.plot(metrics_df['g_gap'], metrics_df['EC50'], 'o-', linewidth=2, markersize=8)
ax.fill_between(metrics_df['g_gap'], 
                metrics_df['EC50_CI_lower'].fillna(metrics_df['EC50']), 
                metrics_df['EC50_CI_upper'].fillna(metrics_df['EC50']), 
                alpha=0.3, label='95% CI')
ax.set_xlabel('Gap Junction Conductance (nS)')
ax.set_ylabel('EC50 (pA)')
ax.set_title('C. Sensitivity (EC50) vs. Coupling')
ax.grid(True, alpha=0.3)
ax.legend()

# Panel D: Hill coefficient (dynamic range) vs. coupling
ax = axes[1, 1]
ax.plot(metrics_df['g_gap'], metrics_df['Hill_n'], 'o-', 
        linewidth=2, markersize=8, color='coral')
ax.set_xlabel('Gap Junction Conductance (nS)')
ax.set_ylabel('Hill Coefficient (slope)')
ax.set_title('D. Dynamic Range (Hill n) vs. Coupling')
ax.grid(True, alpha=0.3)
ax.axhline(y=1, linestyle='--', color='gray', alpha=0.5, label='Linear (n=1)')
ax.legend()

plt.tight_layout()
plt.savefig('coupling_dose_response_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved as 'coupling_dose_response_analysis.png'")

## 8. Visualization: SNR Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel A: SNR heatmap
ax = axes[0]
pivot_snr = df_mean.pivot(index='Intensity', columns='g_gap', values='SNR')
sns.heatmap(pivot_snr, cmap='YlOrRd', ax=ax, cbar_kws={'label': 'SNR'})
ax.set_xlabel('Gap Conductance (nS)')
ax.set_ylabel('Flash Intensity (pA)')
ax.set_title('A. SNR Heatmap')

# Panel B: SNR at EC50 vs. coupling
ax = axes[1]
ax.plot(metrics_df['g_gap'], metrics_df['SNR_at_EC50'], 'o-', 
        linewidth=2, markersize=8, color='darkgreen')
ax.set_xlabel('Gap Junction Conductance (nS)')
ax.set_ylabel('SNR at EC50')
ax.set_title('B. SNR at Half-Maximal Response')
ax.grid(True, alpha=0.3)

# Panel C: Total SNR (AUC) vs. coupling
ax = axes[2]
ax.plot(metrics_df['g_gap'], metrics_df['SNR_AUC'], 'o-', 
        linewidth=2, markersize=8, color='purple')
ax.set_xlabel('Gap Junction Conductance (nS)')
ax.set_ylabel('SNR Area Under Curve')
ax.set_title('C. Overall SNR Across Dynamic Range')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('coupling_snr_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved as 'coupling_snr_analysis.png'")

## 9. Summary Report and Interpretation

In [ ]:
# Generate comprehensive summary report
report = f"""
{'='*70}
GAP JUNCTION COUPLING STUDY - SUMMARY REPORT
{'='*70}

EXPERIMENTAL PARAMETERS:
- Gap conductance range: {metrics_df['g_gap'].min():.2f} - {metrics_df['g_gap'].max():.2f} nS
- Number of coupling values tested: {len(metrics_df)}
- Trials per condition: {df.groupby(['g_gap', 'Intensity']).size().max()}
- Stimulus intensities: {len(df_mean['Intensity'].unique())} levels
- Poisson noise: {'Enabled' if POISSON_NOISE_ENABLED else 'Disabled'}
- Gaussian noise: {'Enabled' if GAUSSIAN_NOISE_ENABLED else 'Disabled'}

KEY FINDINGS:

1. SENSITIVITY (EC50):
   - Range: {metrics_df['EC50'].min():.1f} - {metrics_df['EC50'].max():.1f} pA
   - Optimal coupling (lowest EC50): g = {metrics_df.loc[metrics_df['EC50'].idxmin(), 'g_gap']:.2f} nS
   - Effect: {'Decreases' if metrics_df['EC50'].iloc[-1] < metrics_df['EC50'].iloc[0] else 'Increases'} with coupling

2. DYNAMIC RANGE (Hill coefficient):
   - Range: {metrics_df['Hill_n'].min():.2f} - {metrics_df['Hill_n'].max():.2f}
   - Steepest response: g = {metrics_df.loc[metrics_df['Hill_n'].idxmax(), 'g_gap']:.2f} nS
   - Interpretation: {'Compressed' if metrics_df['Hill_n'].mean() > 1 else 'Expanded'} dynamic range

3. SIGNAL-TO-NOISE RATIO:
   - SNR at EC50 range: {metrics_df['SNR_at_EC50'].min():.2f} - {metrics_df['SNR_at_EC50'].max():.2f}
   - Best SNR: g = {metrics_df.loc[metrics_df['SNR_at_EC50'].idxmax(), 'g_gap']:.2f} nS
   - Total SNR (AUC) optimum: g = {metrics_df.loc[metrics_df['SNR_AUC'].idxmax(), 'g_gap']:.2f} nS

4. SATURATION:
   - Saturation intensity range: {metrics_df['saturation_intensity'].min():.1f} - {metrics_df['saturation_intensity'].max():.1f} pA
   - Maximum response range: {metrics_df['R_max'].min():.1f} - {metrics_df['R_max'].max():.1f} Hz

INTERPRETATION:
"""

# Add interpretation based on trends
ec50_trend = "decreases" if metrics_df['EC50'].corr(metrics_df['g_gap']) < 0 else "increases"
snr_trend = "increases" if metrics_df['SNR_at_EC50'].corr(metrics_df['g_gap']) > 0 else "decreases"

report += f"""
- Gap junction coupling {ec50_trend} sensitivity (EC50)
- SNR {snr_trend} with coupling, suggesting {'improved' if snr_trend=='increases' else 'reduced'} signal pooling
- {'Nonmonotonic' if abs(metrics_df['SNR_AUC'].diff().std()) > metrics_df['SNR_AUC'].diff().mean() else 'Monotonic'} relationship suggests tradeoff between sensitivity and dynamic range

CLINICAL RELEVANCE:
- Optimal coupling for achromatopsia: g ≈ {metrics_df.loc[metrics_df['SNR_at_EC50'].idxmax(), 'g_gap']:.2f} nS (maximizes SNR)
- Therapeutic target range: {metrics_df['g_gap'].quantile(0.25):.2f} - {metrics_df['g_gap'].quantile(0.75):.2f} nS

FILES GENERATED:
- coupling_dose_response_analysis.png
- coupling_snr_analysis.png
- coupling_metrics_summary.csv

{'='*70}
"""

print(report)

# Save metrics to CSV
metrics_df.to_csv('coupling_metrics_summary.csv', index=False)
print("\nMetrics saved to 'coupling_metrics_summary.csv'")

# Save full report to text file
with open('coupling_analysis_report.txt', 'w') as f:
    f.write(report)
print("Report saved to 'coupling_analysis_report.txt'")

## 10. Run NEURON Simulation (Optional)

Uncomment and run the cell below to execute the NEURON simulation directly from this notebook. This will take 1-2 hours to complete.

In [ ]:
# UNCOMMENT TO RUN NEURON SIMULATION
# WARNING: This will take 1-2 hours to complete

# import neuron
# from neuron import h
# 
# # Load simulation protocol
# h.load_file('simulation_protocol.hoc')
# 
# print("Network initialized")
# print(f"Rods: {h.rodtotal}")
# print(f"Bipolars: {h.biptotal}")
# print(f"Ganglion cells: {h.gantotal}")
# 
# # Run main coupling sweep experiment
# print("\n" + "="*60)
# print("Starting gap junction coupling sweep...")
# print("Expected runtime: ~2 hours")
# print("="*60 + "\n")
# 
# h.protocol_coupling_sweep()
# 
# print("\n" + "="*60)
# print("Simulation complete!")
# print("Results saved to results_coupling_sweep_*.dat")
# print("Run analysis cells above to process results")
# print("="*60)